In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import torch
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import json
import subprocess
import re
import time
import threading
import io
from fastapi import FastAPI, UploadFile, File
from typing import List  
import uvicorn
from concurrent.futures import ThreadPoolExecutor

In [ ]:
# Instancia o app FastAPI
app = FastAPI()

# Transformações padrão de imagem
image_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.CenterCrop(336),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [ ]:
class CarClassificationModel:
    def __init__(self, model_path, mapping_path=None):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)

        # Detectar número de classes
        num_classes = checkpoint.get("num_classes", 742)
        print(f"✅ Modelo detectado com {num_classes} classes")

        # Criar arquitetura base
        self.model = models.efficientnet_b3(weights=None)
        num_features = self.model.classifier[1].in_features
        self.model.classifier = torch.nn.Sequential(
            torch.nn.Dropout(p=0.3),
            torch.nn.Linear(num_features, 512),
            torch.nn.ReLU(inplace=True),
            torch.nn.BatchNorm1d(512),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(512, num_classes)
        )

        # Carregar pesos do checkpoint
        self.model.load_state_dict(checkpoint["model_state_dict"], strict=False)
        self.model = self.model.to(self.device)
        self.model.eval()

        # ✅ Carregar mapeamento das classes
        if "idx_to_model" in checkpoint:
            print("🔍 Mapeamento carregado direto do checkpoint")
            self.idx_to_model = checkpoint["idx_to_model"]
        elif mapping_path:
            print("📁 Mapeamento carregado do arquivo JSON")
            with open(mapping_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                model_to_idx = data["model_to_idx"]
                self.idx_to_model = {v: k for k, v in model_to_idx.items()}
        else:
            raise ValueError("❌ Nenhum mapeamento encontrado no checkpoint nem no arquivo JSON.")

    def predict(self, image: Image.Image, top_k: int = 3):
        try:
            if image.mode != 'RGB':
                image = image.convert('RGB')

            input_tensor = image_transforms(image).unsqueeze(0).to(self.device)

            with torch.no_grad():
                outputs = self.model(input_tensor)
                probabilities = F.softmax(outputs, dim=1)

            top_probs, top_indices = torch.topk(probabilities, top_k)
            predictions = []

            for i in range(top_k):
                class_idx = top_indices[0][i].item()
                prob = top_probs[0][i].item()

                class_name = self.idx_to_model.get(class_idx, "Desconhecido")

                if "_" in class_name:
                    brand, model = class_name.split("_", 1)
                else:
                    brand, model = class_name, "Desconhecido"

                predictions.append({
                    "rank": i + 1,
                    "class_index": class_idx,
                    "brand": brand,
                    "model": model,
                    "confidence": round(prob * 100, 2)
                })

            return {"success": True, "predictions": predictions}

        except Exception as e:
            return {"success": False, "error": str(e)}

import gdown

In [ ]:
# LINKS DO DRIVE
MODEL_FILE_ID = "1fQPFojTSoI2H2CXxvZZ548ybXSEuv85d"

# Caminhos locais
MODEL_PATH = "best_model_efficientnet_b3_acc_84.81.pth"
MAPPING_PATH = "class_mappings.json"

# Download direto
gdown.download(f"https://drive.google.com/uc?id={MODEL_FILE_ID}", MODEL_PATH, quiet=False)

# Instancia o modelo
car_model = CarClassificationModel(MODEL_PATH, MAPPING_PATH)

# ✅ Inicializar executor para batch
executor = ThreadPoolExecutor(max_workers=8)

In [ ]:
# ============================
# ENDPOINT: Predição simples
# ============================
@app.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    """Identificar um único carro"""
    image_bytes = await file.read()
    img = Image.open(io.BytesIO(image_bytes))
    result = car_model.predict(img)
    return result

# ============================
# ENDPOINT: Predição em batch
# ============================
@app.post("/predict_batch")
async def predict_batch_endpoint(files: List[UploadFile] = File(...)):  # ✅ List do typing
    """
    Endpoint para identificar múltiplos carros em paralelo
    """
    start_time = time.time()

    print(f"🚗 Recebido batch com {len(files)} arquivo(s)")

    try:
        # Validar se há arquivos
        if not files or len(files) == 0:
            return {
                "success": False,
                "error": "Nenhum arquivo foi enviado",
                "predictions": []
            }

        def process(img_bytes):
            """Processar uma imagem individual"""
            try:
                img = Image.open(io.BytesIO(img_bytes))
                # Converte para RGB se estiver em RGBA ou modo diferente
                if img.mode != 'RGB':
                    img = img.convert('RGB')

                # Fazer a predição
                prediction = car_model.predict(img)
                print(f"✅ Imagem processada: {prediction}")

                return prediction
            except Exception as e:
                print(f"❌ Erro ao processar imagem: {str(e)}")
                return {
                    "success": False,
                    "predictions": [{
                        "brand": "Erro",
                        "model": "Não foi possível identificar",
                        "confidence": 0
                    }]
                }

        # Ler todos os bytes dos arquivos
        images_bytes = []
        for f in files:
            try:
                content = await f.read()
                images_bytes.append(content)
                print(f"📥 Arquivo lido: {len(content)} bytes")
            except Exception as e:
                print(f"❌ Erro ao ler arquivo {f.filename}: {str(e)}")

        if not images_bytes:
            return {
                "success": False,
                "error": "Nenhuma imagem pôde ser lida",
                "predictions": []
            }

        # Processar em paralelo usando ThreadPoolExecutor
        print(f"⚙️ Processando {len(images_bytes)} imagens em paralelo...")
        futures = [executor.submit(process, b) for b in images_bytes]
        predictions = []

        for future in futures:
            try:
                result = future.result(timeout=60)  # Timeout de 60 segundos
                predictions.append(result)
            except Exception as e:
                print(f"❌ Erro ao processar predição: {str(e)}")
                predictions.append({
                    "success": False,
                    "predictions": [{
                        "brand": "Erro",
                        "model": "Timeout na predição",
                        "confidence": 0
                    }]
                })

        total_time = time.time() - start_time

        response = {
            "success": True,
            "num_images": len(files),
            "predictions": predictions,
            "execution_time_sec": round(total_time, 2)
        }

        print(f"✅ Batch concluído em {total_time:.2f}s com {len(predictions)} imagens processadas")
        return response

    except Exception as e:
        print(f"❌ Erro geral no batch: {str(e)}")
        import traceback
        traceback.print_exc()
        return {
            "success": False,
            "error": str(e),
            "predictions": []
        }


In [ ]:
# ============================
# Debug endpoint
# ============================
@app.get("/debug/routes")
async def debug_routes():
    """Listar todos os endpoints registrados"""
    routes = []
    for route in app.routes:
        routes.append({
            "path": route.path,
            "methods": getattr(route, 'methods', [])
        })
    return {
        "total_routes": len(routes),
        "routes": routes
    }

# ============================
# Iniciar servidor
# ============================
port = 8000
print("🌐 Iniciando Cloudflare Tunnel...")

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

public_url = None
timeout = time.time() + 90

while True:
    line = process.stderr.readline().decode("utf-8")
    if line:
        print("🌀", line.strip())
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print(f"\n✅ Túnel ativo e funcional:")
            print(f"   📍 /predict → {public_url}/predict")
            print(f"   📍 /predict_batch → {public_url}/predict_batch")
            print(f"   📍 /debug/routes → {public_url}/debug/routes\n")
            break
    if time.time() > timeout:
        print("❌ Timeout: não foi possível obter a URL pública.")
        break

def start_uvicorn():
    config = uvicorn.Config(app=app, host="0.0.0.0", port=port, log_level="info")
    server = uvicorn.Server(config)
    server.run()

thread = threading.Thread(target=start_uvicorn, daemon=True)
thread.start()

# Manter o Colab rodando
import time as time_module
try:
    while True:
        time_module.sleep(1)
except KeyboardInterrupt:
    print("\n⛔ Servidor interrompido")